In [35]:
from collections import deque
import heapq
import os
import sys
import time
import tracemalloc
import pandas as pd
import numpy as np

# --- 0. CORRECCIÓN DE RUTA PARA RECONOCER LA CARPETA SRC ---
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print(f"📁 Directorio de trabajo actual: {os.getcwd()}")

# --- IMPORTACIÓN DE CLASES MODULARES DE LAS 10 ETAPAS DESDE src/ ---
from src.etapa_0_ingesta_buffer import IngestaBufferFIFO
from src.etapa_1_preprocesamiento_lotes import PreprocesamientoLotes
from src.etapa_2_cola_prioridades import ColaPrioridadesHeap
from src.etapa_3_almacenamiento_lookup_cache import AlmacenamientoLookupCache
from src.etapa_4_indexacion_principal import IndexacionPrincipalBPlusTree
from src.etapa_5_indice_multidimensional import IndiceMultidimensionalKDTree
from src.etapa_6_filtrado_probabilistico import BloomFilter, CountMinSketch
from src.etapa_7_similitud_aproximada import (
    HNSWApproximateSearch,
    MinHashLSHSimulator,
)
from src.etapa_8_grafo_dominio import GrafoDominioRed
from src.etapa_9_servicio_visualizacion import TopKAnalytics, TrieAutocompletado

# --- 1. VERIFICACIÓN Y CARGA DE DATOS ---
target_file = 'data/ecommerce_1K.parquet'
print(f'📁 Cargando dataset de prueba desde: {target_file}')
df = pd.read_parquet(target_file)
print(
    f'✅ Dataset cargado con éxito. Total de registros: {len(df):,}\n'
    '------------------------------------------------------------'
)

📁 Directorio de trabajo actual: c:\Users\Jesus\proyecto-eda
📁 Cargando dataset de prueba desde: data/ecommerce_1K.parquet
✅ Dataset cargado con éxito. Total de registros: 1,000
------------------------------------------------------------


In [50]:
#df
#df_duplicados = df[df.duplicated(subset=['product_id'], keep='first')]
#display(df_duplicados.head())
df_duplicado_especifico = df[df['product_id'] == 16700826]
display(df_duplicado_especifico)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,event_id,is_fraud,ip_address,priority
31,2019-11-01 00:00:12 UTC,view,16700826,2053013559901684381,furniture.kitchen.chair,nika,47.77,566255262,173d7b72-1db7-4638-8207-be8148bf3c9f,c85cd135,False,192.168.26.41,5
55,2019-11-01 00:00:18 UTC,view,16700826,2053013559901684381,furniture.kitchen.chair,nika,47.77,566255262,173d7b72-1db7-4638-8207-be8148bf3c9f,5c203346,False,192.168.3.65,5


In [36]:
# ============================================================
# ETAPA 0: Ingesta y Búfer Continuo (Stream a Memoria)
# ============================================================
print('--- [ETAPA 0] Ingesta y Búfer FIFO (Queue) ---')
tracemalloc.start()
start_time = time.perf_counter()

buffer = IngestaBufferFIFO(capacity=5000)
for _, row in df.iterrows():
    buffer.enqueue(row.to_dict())

lote_crudo = [buffer.dequeue() for _ in range(buffer.size())]

end_time = time.perf_counter()
_, peak_mem_etapa0 = tracemalloc.get_traced_memory()
tracemalloc.stop()
tiempo_etapa0 = (end_time - start_time) * 1000

print(f'    > Elementos en búfer FIFO: {buffer.size()}')
print(f'    > Lote crudo total extraído: {len(lote_crudo)} elementos')
print(f'    ⏱️ Tiempo de ejecución: {tiempo_etapa0:.4f} ms')
print(
    f'    💾 Memoria RAM pico: {peak_mem_etapa0 / 1024:.2f} KB\n'
    '------------------------------------------------------------'
)

--- [ETAPA 0] Ingesta y Búfer FIFO (Queue) ---
    > Elementos en búfer FIFO: 0
    > Lote crudo total extraído: 1000 elementos
    ⏱️ Tiempo de ejecución: 644.3373 ms
    💾 Memoria RAM pico: 1683.15 KB
------------------------------------------------------------


In [37]:
# ============================================================
# ETAPA 1: Preprocesamiento de Lotes (Limpieza y Deduplicación)
# ============================================================
print('--- [ETAPA 1] Preprocesamiento de Lotes (Timsort / Búsqueda Binaria) ---')
tracemalloc.start()
start_time = time.perf_counter()

preproc = PreprocesamientoLotes()
lote_dedup = preproc.deduplicate_batch(lote_crudo)
lote_ordenado = preproc.sort_batch(lote_dedup, key='event_time')

end_time = time.perf_counter()
_, peak_mem_etapa1 = tracemalloc.get_traced_memory()
tracemalloc.stop()
tiempo_etapa1 = (end_time - start_time) * 1000

print(f'    > Elementos limpios y ordenados: {len(lote_ordenado)}')
print(f'    ⏱️ Tiempo de ejecución: {tiempo_etapa1:.4f} ms')
print(
    f'    💾 Memoria RAM pico: {peak_mem_etapa1 / 1024:.2f} KB\n'
    '------------------------------------------------------------'
)

--- [ETAPA 1] Preprocesamiento de Lotes (Timsort / Búsqueda Binaria) ---
    > Elementos limpios y ordenados: 1000
    ⏱️ Tiempo de ejecución: 31.5375 ms
    💾 Memoria RAM pico: 45.40 KB
------------------------------------------------------------


In [38]:
# ============================================================
# ETAPA 2: Cola de Prioridades de Negocio (MinHeap)
# ============================================================
print('--- [ETAPA 2] Cola de Prioridades de Negocio (MinHeap) ---')
tracemalloc.start()
start_time = time.perf_counter()

cola_prioridad = ColaPrioridadesHeap()
for ev in lote_ordenado:
    ev['priority'] = 1 if ev.get('event_type') == 'purchase' else 3
    cola_prioridad.push(ev)

lote_priorizado = [cola_prioridad.pop() for _ in range(len(lote_ordenado))]

end_time = time.perf_counter()
_, peak_mem_etapa2 = tracemalloc.get_traced_memory()
tracemalloc.stop()
tiempo_etapa2 = (end_time - start_time) * 1000

print(f'    > Eventos priorizados en MinHeap: {cola_prioridad.size()}')
print(f'    ⏱️ Tiempo de ejecución: {tiempo_etapa2:.4f} ms')
print(
    f'    💾 Memoria RAM pico: {peak_mem_etapa2 / 1024:.2f} KB\n'
    '------------------------------------------------------------'
)

--- [ETAPA 2] Cola de Prioridades de Negocio (MinHeap) ---
    > Eventos priorizados en MinHeap: 0
    ⏱️ Tiempo de ejecución: 205.6559 ms
    💾 Memoria RAM pico: 128.00 KB
------------------------------------------------------------


In [43]:
import pandas as pd
from src.etapa_0_ingesta_buffer import IngestaBufferFIFO
from src.etapa_6_filtrado_probabilistico import BloomFilter

# 1. Cargar el dataset base de 1K
df = pd.read_parquet('data/ecommerce_1K.parquet')
elementos = df.to_dict(orient='records')
total_inicial = len(elementos)

# 2. Simular la ingesta y el filtrado exacto de la Etapa 6
bloom = BloomFilter(capacity=total_inicial, error_rate=0.01)
lote_filtrado = []

for ev in elementos:
    prod_id = str(ev.get('product_id'))
    is_duplicate = prod_id in bloom
    bloom.add(prod_id)
    if not is_duplicate:
        lote_filtrado.append(ev)

total_final = len(lote_filtrado)
total_filtrados = total_inicial - total_final

print("==========================================")
print("📊 VERIFICACIÓN MATEMÁTICA DE FLUJO (ETAPA 6)")
print("==========================================")
print(f"🔹 Eventos iniciales ingresados : {total_inicial}")
print(f"🔹 Eventos que pasaron limpios  : {total_final}")
print(f"🔹 Elementos filtrados/duplicados: {total_filtrados}")
print("==========================================")

📊 VERIFICACIÓN MATEMÁTICA DE FLUJO (ETAPA 6)
🔹 Eventos iniciales ingresados : 1000
🔹 Eventos que pasaron limpios  : 672
🔹 Elementos filtrados/duplicados: 328
